# GMM Region Finder

Automatically detects region boundaries in a PCA projection of a Shapley behavioral space
using Gaussian Mixture Models (GMM) with a BIC elbow criterion.

**Workflow:**
1. Load pre-computed behavioral spaces (`.npy` file from `behavioral_space_explorer.py`)
2. Project the chosen space to 2D via PCA
3. Fit GMMs to the PC1 and PC2 marginal distributions independently
4. Retain boundaries only where the GMM density falls below a threshold fraction of the peak
5. Display diagnostic plots so you can verify or override the detected boundaries
6. Generate a `USER_REGIONS` dict ready to pass to `behavioral_region_explorer.py`

**Prerequisites:** Run `behavioral_space_explorer.py` first to produce the `.npy` behavioral spaces file.

In [ ]:
# ============================================================
# CONFIGURATION — edit these for your dataset
# ============================================================

# Path to the .npy file produced by behavioral_space_explorer.py
BEHAVIORAL_SPACES_FILE = 'behavioral_exploration/mxene_behavioral_spaces.npy'

# Which behavioral space to analyse  (variance | skewness | kurtosis | entropy)
SPACE_NAME = 'skewness'

# Dataset name — used in output file names
DATASET_NAME = 'mxene'

# Where to save the diagnostic figure
OUTPUT_DIR = 'behavioral_exploration'

# ── GMM tuning parameters ────────────────────────────────────────────
# Maximum number of Gaussian components to try
K_MAX = 8
# Ignore components whose weight is below this fraction (noise suppression)
MIN_WEIGHT = 0.05
# BIC elbow: stop adding components once the marginal BIC gain falls below
# this fraction of the largest single-step gain
ELBOW_FRACTION = 0.10
# Keep a boundary only if the GMM valley density is below this fraction
# of the global peak density (ensures boundaries sit in genuine low-density gaps)
MAX_VALLEY_DENSITY = 0.10

## Step 1 — Imports and data loading

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from scipy.stats import norm as _norm

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load behavioral spaces
behavioral_spaces = np.load(BEHAVIORAL_SPACES_FILE, allow_pickle=True).item()
print('Loaded behavioral spaces:')
for k, v in behavioral_spaces.items():
    print(f'  {k}: {v.shape}')

# Project the chosen space to 2D
X = behavioral_spaces[SPACE_NAME]
X_scaled = MinMaxScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
pc1 = X_pca[:, 0]
pc2 = X_pca[:, 1]

var_exp = pca.explained_variance_ratio_
print(f'\n{SPACE_NAME.capitalize()} space PCA:')
print(f'  PC1 variance explained: {var_exp[0]:.1%}')
print(f'  PC2 variance explained: {var_exp[1]:.1%}')
print(f'  PC1 range: [{pc1.min():.4f}, {pc1.max():.4f}]')
print(f'  PC2 range: [{pc2.min():.4f}, {pc2.max():.4f}]')


def build_user_regions(pc1_boundaries, pc2_boundaries, pc1_vals, pc2_vals,
                       space_name, palette='tab10'):
    """Build a USER_REGIONS dict from boundary arrays.

    Regions are numbered from low to high along each axis:
      PC1 band 1 = lowest PC1 values, PC2 band 1 = lowest PC2 values.
    """
    pc1_edges = np.concatenate([[pc1_vals.min()],
                                np.sort(pc1_boundaries),
                                [pc1_vals.max()]])
    pc2_edges = np.concatenate([[pc2_vals.min()],
                                np.sort(pc2_boundaries),
                                [pc2_vals.max()]])

    n_pc1   = len(pc1_edges) - 1
    n_pc2   = len(pc2_edges) - 1
    n_total = n_pc1 * n_pc2
    cmap    = plt.cm.get_cmap(palette, max(n_total, 1))

    regions = {}
    idx = 0
    for i in range(n_pc1):
        for j in range(n_pc2):
            name = f'Region_PC1_{i+1}_PC2_{j+1}'
            rgba = cmap(idx)
            color = '#{:02x}{:02x}{:02x}'.format(
                int(rgba[0] * 255), int(rgba[1] * 255), int(rgba[2] * 255))
            regions[name] = {
                'space':       space_name,
                'pc1_range':   (float(pc1_edges[i]),     float(pc1_edges[i + 1])),
                'pc2_range':   (float(pc2_edges[j]),     float(pc2_edges[j + 1])),
                'description': (f'PC1 band {i+1}/{n_pc1}, '
                                f'PC2 band {j+1}/{n_pc2}'),
                'color':       color,
            }
            idx += 1
    return regions

## Step 2 — GMM boundary detection

GMMs are fitted independently to the PC1 and PC2 marginal distributions.
The optimal number of components is selected by a BIC elbow criterion.
A candidate boundary between two adjacent components is kept only if the
GMM density at the crossing point is below `MAX_VALLEY_DENSITY × peak density`.

In [ ]:
def _select_k_elbow(bic_scores, elbow_fraction):
    """Return the smallest k where the next BIC improvement is below elbow_fraction
    of the largest single-step improvement."""
    improvements = -np.diff(bic_scores)
    if len(improvements) == 0 or improvements.max() <= 0:
        return 1
    threshold = elbow_fraction * improvements.max()
    below = np.where(improvements < threshold)[0]
    return int(below[0]) + 1 if len(below) > 0 else len(bic_scores)


def find_boundaries_gmm(values, k_max=8, min_weight=0.05,
                        elbow_fraction=0.10, max_valley_density=0.10):
    """Fit a GMM to 1-D values and return detected boundary positions.

    Returns
    -------
    boundaries : ndarray of shape (n_boundaries,)
    k_opt      : int, selected number of components
    bic        : list of BIC values for k = 1 .. k_max
    model      : fitted GaussianMixture at k_opt
    """
    vals = values.reshape(-1, 1)
    bic, models = [], []
    for k in range(1, k_max + 1):
        gmm = GaussianMixture(n_components=k, covariance_type='full',
                              random_state=42, n_init=10, max_iter=300)
        gmm.fit(vals)
        bic.append(gmm.bic(vals))
        models.append(gmm)

    k_opt = _select_k_elbow(bic, elbow_fraction)
    model = models[k_opt - 1]

    means   = model.means_.flatten()
    stds    = np.sqrt(model.covariances_[:, 0, 0])
    weights = model.weights_.copy()

    # Drop components below the weight threshold
    keep = weights >= min_weight
    means, stds, weights = means[keep], stds[keep], weights[keep]
    if weights.sum() > 0:
        weights /= weights.sum()

    if len(means) <= 1:
        return np.array([]), k_opt, bic, model

    order = np.argsort(means)
    means, stds, weights = means[order], stds[order], weights[order]

    x_grid = np.linspace(values.min(), values.max(), 3000)
    gmm_density = np.exp(model.score_samples(x_grid.reshape(-1, 1)))
    peak_density = gmm_density.max()

    boundary_pts = []
    for i in range(len(means) - 1):
        lo, hi = means[i], means[i + 1]
        seg = x_grid[(x_grid >= lo) & (x_grid <= hi)]
        if len(seg) < 2:
            seg = np.linspace(lo, hi, 300)
        d1 = weights[i]     * _norm.pdf(seg, means[i],     stds[i])
        d2 = weights[i + 1] * _norm.pdf(seg, means[i + 1], stds[i + 1])
        crossings = np.where(np.diff(np.sign(d1 - d2)))[0]
        bp = (0.5 * (seg[crossings[0]] + seg[crossings[0] + 1])
              if len(crossings) > 0 else 0.5 * (means[i] + means[i + 1]))
        valley_d = gmm_density[np.argmin(np.abs(x_grid - bp))]
        if valley_d / peak_density <= max_valley_density:
            boundary_pts.append(float(bp))

    return np.array(boundary_pts), k_opt, bic, model


# ── Run detection ─────────────────────────────────────────────────────
pc1_boundaries, pc1_k, pc1_bic, pc1_model = find_boundaries_gmm(
    pc1, K_MAX, MIN_WEIGHT, ELBOW_FRACTION, MAX_VALLEY_DENSITY)
pc2_boundaries, pc2_k, pc2_bic, pc2_model = find_boundaries_gmm(
    pc2, K_MAX, MIN_WEIGHT, ELBOW_FRACTION, MAX_VALLEY_DENSITY)

print('=' * 54)
print(f'{SPACE_NAME.upper()} SPACE  —  GMM BOUNDARY DETECTION')
print('=' * 54)
print(f'PC1: elbow k={pc1_k}  ->  {len(pc1_boundaries)} boundary(ies)')
for i, v in enumerate(pc1_boundaries, 1):
    print(f'  Boundary {i}: PC1 = {v:.6f}')
print(f'PC2: elbow k={pc2_k}  ->  {len(pc2_boundaries)} boundary(ies)')
for i, v in enumerate(pc2_boundaries, 1):
    print(f'  Boundary {i}: PC2 = {v:.6f}')
print(f'Grid: {len(pc1_boundaries)+1} x {len(pc2_boundaries)+1}'
      f' = {(len(pc1_boundaries)+1)*(len(pc2_boundaries)+1)} regions')

# ── Build USER_REGIONS from auto-detected boundaries ──────────────────
USER_REGIONS = build_user_regions(
    pc1_boundaries, pc2_boundaries, pc1, pc2, SPACE_NAME)

print()
print('# ── USER_REGIONS (auto-detected) — paste into behavioral_region_explorer.py ──')
print('USER_REGIONS = {')
for name, cfg in USER_REGIONS.items():
    mask = ((pc1 >= cfg['pc1_range'][0]) & (pc1 <= cfg['pc1_range'][1]) &
            (pc2 >= cfg['pc2_range'][0]) & (pc2 <= cfg['pc2_range'][1]))
    print(f"    '{name}': {{")
    print(f"        'space':       '{cfg['space']}',")
    print(f"        'pc1_range':   ({cfg['pc1_range'][0]:.6f}, {cfg['pc1_range'][1]:.6f}),")
    print(f"        'pc2_range':   ({cfg['pc2_range'][0]:.6f}, {cfg['pc2_range'][1]:.6f}),")
    print(f"        'description': '{cfg['description']}',  # n={int(mask.sum())}")
    print(f"        'color':       '{cfg['color']}',")
    print( "    },")
print('}')
print()
print('Review the diagnostic plots (Step 3) and override boundaries in Step 4 if needed.'
      '  USER_REGIONS will be rebuilt in Step 5.')

## Step 3 — Diagnostic plots

Each row shows one PC axis.  
**Left panel:** histogram with fitted GMM overlaid; individual components are coloured curves;
accepted boundaries are red dashed lines; the grey dotted line is the density cutoff.  
**Right panel:** BIC vs number of components; the red dashed line marks the selected elbow.

If a boundary looks wrong (e.g. it splits a dense cluster), adjust `MAX_VALLEY_DENSITY`
downward (stricter) or override it manually in Step 4.

In [ ]:
def _plot_gmm_diagnostic(ax_hist, ax_bic, values, model, boundaries, bic_scores,
                         k_opt, label, min_weight, max_valley_density):
    """Four-panel diagnostic: histogram + BIC curve for one PC axis."""
    x = np.linspace(values.min(), values.max(), 600)
    density_full = np.exp(model.score_samples(x.reshape(-1, 1)))
    peak = density_full.max()

    # Histogram
    ax_hist.hist(values, bins=60, density=True, alpha=0.35, color='steelblue',
                 label='data')

    # Individual components
    colors = plt.cm.tab10(np.linspace(0, 1, model.n_components))
    mu_all = model.means_.flatten()
    sg_all = np.sqrt(model.covariances_[:, 0, 0])
    wt_all = model.weights_
    for j in range(model.n_components):
        ls = '-' if wt_all[j] >= min_weight else ':'
        ax_hist.plot(x, wt_all[j] * _norm.pdf(x, mu_all[j], sg_all[j]),
                     ls, color=colors[j], lw=1.5, alpha=0.75)

    # Full mixture and cutoff line
    ax_hist.plot(x, density_full, 'k-', lw=2, label=f'GMM k={k_opt}')
    ax_hist.axhline(max_valley_density * peak, color='gray', ls=':',
                    lw=1.5, label=f'density cutoff ({max_valley_density:.0%})')

    # Accepted boundaries
    for v in boundaries:
        ax_hist.axvline(v, color='red', ls='--', lw=2)

    ax_hist.set_xlabel(label, fontsize=12)
    ax_hist.set_ylabel('Density', fontsize=11)
    ax_hist.set_title(f'{label}: {len(boundaries)} boundary(ies)  [k={k_opt}]',
                      fontsize=12)
    ax_hist.legend(fontsize=9)
    ax_hist.grid(alpha=0.3)

    # BIC curve
    ks = np.arange(1, len(bic_scores) + 1)
    ax_bic.plot(ks, bic_scores, 'o-', color='black', lw=2)
    ax_bic.axvline(k_opt, color='red', ls='--', lw=2, label=f'elbow k={k_opt}')
    ax_bic.set_xlabel('k (components)', fontsize=12)
    ax_bic.set_ylabel('BIC', fontsize=11)
    ax_bic.set_title(f'{label}: BIC vs k', fontsize=12)
    ax_bic.legend(fontsize=9)
    ax_bic.grid(alpha=0.3)


fig, axes = plt.subplots(2, 2, figsize=(14, 9))
_plot_gmm_diagnostic(axes[0, 0], axes[0, 1],
                     pc1, pc1_model, pc1_boundaries, pc1_bic, pc1_k,
                     'PC1', MIN_WEIGHT, MAX_VALLEY_DENSITY)
_plot_gmm_diagnostic(axes[1, 0], axes[1, 1],
                     pc2, pc2_model, pc2_boundaries, pc2_bic, pc2_k,
                     'PC2', MIN_WEIGHT, MAX_VALLEY_DENSITY)
plt.suptitle(
    f'{DATASET_NAME} — {SPACE_NAME.capitalize()} Space: GMM Boundary Detection',
    fontsize=14)
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR,
    f'{DATASET_NAME}_{SPACE_NAME}_gmm_boundaries.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

## Step 4 — Review and optionally override boundaries

If the automatically detected boundaries look correct in the diagnostic plots above,
leave this cell unchanged and run it as-is.

To override, uncomment the relevant lines and enter your own values.
You can also add or remove boundaries by changing the array contents.

In [ ]:
# ── Optional manual overrides ────────────────────────────────────────
# Uncomment and edit to replace auto-detected values.
# Use an empty array np.array([]) to force no boundaries on an axis.

# pc1_boundaries = np.array([0.35])           # e.g. one split on PC1
# pc2_boundaries = np.array([-0.15, 0.20])    # e.g. two splits on PC2

print('Active PC1 boundaries:', pc1_boundaries)
print('Active PC2 boundaries:', pc2_boundaries)
print(f'Grid will be {len(pc1_boundaries)+1} x {len(pc2_boundaries)+1}'
      f' = {(len(pc1_boundaries)+1)*(len(pc2_boundaries)+1)} regions')

## Step 5 — Build USER_REGIONS

Generates a `USER_REGIONS` dictionary covering every cell of the
PC1 × PC2 boundary grid.  Region names follow the pattern `Region_PC1_{i}_PC2_{j}`
where `i` and `j` are 1-indexed band numbers (low to high).

Rename the keys in the printed output before pasting into `behavioral_region_explorer.py`
if you prefer more descriptive labels (e.g. `C_Region_I`).

In [ ]:
USER_REGIONS = build_user_regions(
    pc1_boundaries, pc2_boundaries, pc1, pc2, SPACE_NAME)

print(f'Generated {len(USER_REGIONS)} region(s):')
print(f'  {len(pc1_boundaries)+1} PC1 band(s) x {len(pc2_boundaries)+1} PC2 band(s)\n')
for name, cfg in USER_REGIONS.items():
    mask = ((pc1 >= cfg['pc1_range'][0]) & (pc1 <= cfg['pc1_range'][1]) &
            (pc2 >= cfg['pc2_range'][0]) & (pc2 <= cfg['pc2_range'][1]))
    n = int(mask.sum())
    print(f'  {name}  n={n}')
    print(f'    PC1: [{cfg["pc1_range"][0]:.4f}, {cfg["pc1_range"][1]:.4f}]  '
          f'PC2: [{cfg["pc2_range"][0]:.4f}, {cfg["pc2_range"][1]:.4f}]')

## Step 6 — Print USER_REGIONS as copy-pasteable Python

Copy the output below into your `behavioral_region_explorer.py` session,
renaming the region keys as desired.

In [ ]:
print('# ── paste into behavioral_region_explorer.py ──────────────────')
print('USER_REGIONS = {')
for name, cfg in USER_REGIONS.items():
    print(f"    '{name}': {{")
    print(f"        'space':       '{cfg['space']}',")
    print(f"        'pc1_range':   ({cfg['pc1_range'][0]:.6f}, {cfg['pc1_range'][1]:.6f}),")
    print(f"        'pc2_range':   ({cfg['pc2_range'][0]:.6f}, {cfg['pc2_range'][1]:.6f}),")
    print(f"        'description': '{cfg['description']}',")
    print(f"        'color':       '{cfg['color']}',")
    print( "    },")
print('}')

## Step 7 — Quick 2-D scatter preview

Visualises the detected region grid directly on the PCA scatter so you can
verify region boundaries before running the full analysis.

In [ ]:
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pc1, pc2, c='lightgrey', s=20, alpha=0.6, edgecolors='none',
           zorder=1, label='all samples')

for name, cfg in USER_REGIONS.items():
    mask = ((pc1 >= cfg['pc1_range'][0]) & (pc1 <= cfg['pc1_range'][1]) &
            (pc2 >= cfg['pc2_range'][0]) & (pc2 <= cfg['pc2_range'][1]))
    ax.scatter(pc1[mask], pc2[mask], c=cfg['color'], s=30, alpha=0.7,
               edgecolors='k', linewidths=0.4, zorder=2, label=f"{name} (n={mask.sum()})")
    p1lo, p1hi = cfg['pc1_range']
    p2lo, p2hi = cfg['pc2_range']
    rect = Rectangle((p1lo, p2lo), p1hi - p1lo, p2hi - p2lo,
                      linewidth=1.8, edgecolor=cfg['color'],
                      facecolor='none', linestyle='--', zorder=3)
    ax.add_patch(rect)

ax.set_xlabel(f'PC1 ({var_exp[0]:.1%})', fontsize=13)
ax.set_ylabel(f'PC2 ({var_exp[1]:.1%})', fontsize=13)
ax.set_title(f'{DATASET_NAME} — {SPACE_NAME.capitalize()} space: GMM regions',
             fontsize=13)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
preview_path = os.path.join(OUTPUT_DIR,
    f'{DATASET_NAME}_{SPACE_NAME}_gmm_regions_preview.png')
plt.savefig(preview_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {preview_path}')

## Step 8 — Run behavioral_region_explorer.py  *(optional)*

If this notebook is in the same directory as `behavioral_region_explorer.py`,
fill in the dataset variables below and uncomment `%run` to run the full
region analysis immediately without copying the `USER_REGIONS` dict manually.

The variables set in Steps 1–6 above are already in scope; only the
dataset-specific variables listed below need to be added.

In [ ]:
# Fill in these variables for your dataset, then uncomment %run.

# DATA_FILE    = 'mxene_mendeleev.csv'
# ID_COLUMN    = 'Formula'
# DROP_COLUMNS = ['In-plane_lattice', 'Intercalated_lattice', 'ID']
# LABEL_COLUMNS = ['Voltage', 'Capacity', 'Charge', 'M', 'X', 'T', 'Z']
# PLOT_MODE    = 'combined'

# %run -i behavioral_region_explorer.py

---

# Part 2 — K-means Assisted Boundary Finder

## When to use this method

Use the **k-means finder** when the PCA projection shows a large central bulk with
one or more satellite clusters radiating outward as distinct arms, for example when
a small subset of compositions has extreme feature values along one axis only.

The **GMM finder (Part 1)** is better suited to distributions with parallel bands
separated cleanly along a single PC axis (e.g. a chemical-family split that runs
across the full PC1 range without a dominant central core).

The two methods can be run independently; only one needs to be passed to
`behavioral_region_explorer.py`.

## Algorithm

1. KMeans (k clusters) is applied jointly to the 2-D PC1/PC2 projection.
2. The largest cluster is labelled the **bulk**.
3. Each remaining cluster is classified as a **PC1 arm** (centroid deviates more
   in PC1 from the bulk centroid) or a **PC2 arm** (deviates more in PC2).
4. Rectangular boundaries are computed from data-edge midpoints between adjacent
   clusters. If cluster edges overlap, the boundary is placed just past the lower
   cluster's maximum and a warning is printed.
5. A `USER_REGIONS_KM` dict is generated with region names
   `{KM_DATASET_LABEL}_A`, `_B`, `_C`, … ordered from low to high PC1 then
   along the PC2 arm.

## Tuning

| Parameter | Effect | Default |
|---|---|---|
| `K` | Number of clusters | `4` |
| `KM_SPACE_NAME` | Which behavioral space to project | same as Part 1 |
| `KM_COLORS` | Region colours (one per cluster) | matplotlib tab10 subset |
| `KM_DATASET_LABEL` | Prefix for region dict keys | `DATASET_NAME` |

In [ ]:
# ============================================================
# K-MEANS CONFIGURATION — edit these for your dataset
# ============================================================

# Space to analyse — reuse SPACE_NAME from Part 1, or change independently
KM_SPACE_NAME = SPACE_NAME   # 'variance' | 'skewness' | 'kurtosis' | 'entropy'

# Number of clusters (typically 3-5; try K=3 first, increase if needed)
K = 4

# Colours — must have at least K entries
KM_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# Region label prefix used in the USER_REGIONS_KM dict keys
KM_DATASET_LABEL = DATASET_NAME

## Step K1 — Clustering and boundary computation

Run the cell below to:
1. Apply KMeans to the 2-D PC projection.
2. Classify each non-bulk cluster as a **PC1 arm** or **PC2 arm** based on which axis its centroid deviates from the bulk centroid.
3. Compute rectangular boundaries as midpoints between the data edges of adjacent clusters.
4. Build `USER_REGIONS_KM`, ready for `behavioral_region_explorer.py`.

If a cluster is misidentified (e.g. an arm is classed as bulk), try increasing `K` by 1 and re-running.

In [ ]:
from sklearn.cluster import KMeans

# ── Load and project space (reuses Part 1 projection if same space) ──
if KM_SPACE_NAME == SPACE_NAME:
    km_pc1, km_pc2 = pc1.copy(), pc2.copy()
    km_var_exp = var_exp
else:
    _km_X = behavioral_spaces[KM_SPACE_NAME]
    _km_Xs = MinMaxScaler().fit_transform(_km_X)
    _km_pca = PCA(n_components=2, random_state=42)
    _km_Xp = _km_pca.fit_transform(_km_Xs)
    km_pc1, km_pc2 = _km_Xp[:, 0], _km_Xp[:, 1]
    km_var_exp = _km_pca.explained_variance_ratio_
    print(f'{KM_SPACE_NAME.capitalize()} space: '
          f'PC1={km_var_exp[0]:.1%}, PC2={km_var_exp[1]:.1%}')

km_X2d = np.column_stack([km_pc1, km_pc2])
km_pc1_min = round(km_pc1.min(), 4);  km_pc1_max = round(km_pc1.max(), 4)
km_pc2_min = round(km_pc2.min(), 4);  km_pc2_max = round(km_pc2.max(), 4)

# ── KMeans ────────────────────────────────────────────────────────────
km_fit = KMeans(n_clusters=K, random_state=42, n_init=20)
km_labels = km_fit.fit_predict(km_X2d)
km_centroids = km_fit.cluster_centers_
km_sizes = np.bincount(km_labels)

# ── Cluster role identification ────────────────────────────────────────
km_bulk_id = int(np.argmax(km_sizes))
km_bulk_c  = km_centroids[km_bulk_id]

km_pc1_arm, km_pc2_arm = [], []
for _i in [_j for _j in range(K) if _j != km_bulk_id]:
    _pc1_dev = abs(km_centroids[_i, 0] - km_bulk_c[0])
    _pc2_dev = abs(km_centroids[_i, 1] - km_bulk_c[1])
    (km_pc1_arm if _pc1_dev >= _pc2_dev else km_pc2_arm).append(_i)

km_pc1_arm.sort(key=lambda _i: km_centroids[_i, 0])
km_pc2_arm.sort(key=lambda _i: km_centroids[_i, 1], reverse=True)

# ── Boundary helper ───────────────────────────────────────────────────
def _data_edge_midpt(lo_mask, hi_mask, axis, capture_hi_bottom=False):
    lo_vals   = km_X2d[lo_mask, axis]
    hi_vals   = km_X2d[hi_mask, axis]
    lo_max    = lo_vals.max()
    hi_sorted = np.sort(hi_vals)
    ax_name   = 'PC1' if axis == 0 else 'PC2'
    if capture_hi_bottom:
        gap_mid  = ((hi_sorted[0] + hi_sorted[1]) / 2
                    if len(hi_sorted) >= 2 else hi_sorted[0] + 0.05)
        boundary = round(max(gap_mid, lo_max + 1e-4), 4)
        n_in_lo  = int((hi_vals < boundary).sum())
        if n_in_lo > 0:
            print(f'  NOTE ({ax_name}): boundary={boundary:.4f}; '
                  f'{n_in_lo} point(s) reassigned.')
        return boundary
    hi_min = hi_sorted[0]
    if lo_max < hi_min:
        return round((lo_max + hi_min) / 2, 4)
    else:
        boundary = round(lo_max + 1e-3, 4)
        n_spill  = int((hi_vals < boundary).sum())
        if n_spill > 0:
            print(f'  WARNING ({ax_name}): overlap; boundary={boundary:.4f}; '
                  f'{n_spill} point(s) spill into lower region.')
        return boundary

# ── PC1 splits (across the full data range) ───────────────────────────
km_top_chain = km_pc1_arm + [km_bulk_id]
km_pc1_splits = [
    _data_edge_midpt(km_labels == km_top_chain[j],
                     km_labels == km_top_chain[j + 1], axis=0)
    for j in range(len(km_top_chain) - 1)
]

# Reconcile points that crossed a PC1 boundary after re-labelling
km_labels = km_labels.copy()
for j in range(len(km_top_chain) - 1):
    lo_id, hi_id = km_top_chain[j], km_top_chain[j + 1]
    km_labels[(km_labels == hi_id) & (km_pc1 < km_pc1_splits[j])] = lo_id
km_sizes = np.bincount(km_labels)

km_pc1_edges = [km_pc1_min] + km_pc1_splits + [km_pc1_max]
km_pc1_corner = km_pc1_splits[-1] if km_pc1_splits else km_pc1_min

# ── PC2 floors per PC1 band ───────────────────────────────────────────
km_pc2_bottoms = []
km_bulk_pc2_boundary = km_pc2_min

for j, cid in enumerate(km_top_chain):
    x0, x1 = km_pc1_edges[j], km_pc1_edges[j + 1]
    in_band = (km_pc1 >= x0) & (km_pc1 <= x1)
    if cid == km_bulk_id and km_pc2_arm:
        lo_in = in_band & np.isin(km_labels, km_pc2_arm)
        hi_in = in_band & (km_labels == km_bulk_id)
        if lo_in.sum() >= 1 and hi_in.sum() >= 1:
            b = _data_edge_midpt(lo_in, hi_in, axis=1,
                                 capture_hi_bottom=(len(km_pc2_arm) == 1))
        elif hi_in.sum() > 0:
            b = round(km_pc2[hi_in].min(), 4)
        else:
            b = km_pc2_min
        km_bulk_pc2_boundary = b
    else:
        cib = in_band & (km_labels == cid)
        b = round(km_pc2[cib].min(), 4) if cib.sum() > 0 else km_pc2_min
    km_pc2_bottoms.append(b)

km_pc2_inner_splits = []
for j in range(len(km_pc2_arm) - 1):
    lo_id, hi_id = km_pc2_arm[j + 1], km_pc2_arm[j]
    km_pc2_inner_splits.append(
        _data_edge_midpt(km_labels == lo_id, km_labels == hi_id, axis=1,
                         capture_hi_bottom=(j == len(km_pc2_arm) - 2))
    )

km_pc2_arm_edges = ([km_bulk_pc2_boundary]
                    + km_pc2_inner_splits + [km_pc2_min])

# ── Build USER_REGIONS_KM ────────────────────────────────────────────
km_letters    = [chr(65 + j) for j in range(K)]
km_ordered_ids = km_top_chain + km_pc2_arm
USER_REGIONS_KM = {}

for j, cid in enumerate(km_top_chain):
    lbl = f'{KM_DATASET_LABEL}_{km_letters[j]}'
    USER_REGIONS_KM[lbl] = {
        'space':       KM_SPACE_NAME,
        'pc1_range':   (km_pc1_edges[j], km_pc1_edges[j + 1]),
        'pc2_range':   (km_pc2_bottoms[j], km_pc2_max),
        'description': ('bulk' if cid == km_bulk_id
                        else f'PC1-arm cluster {km_letters[j]}'),
        'color':       KM_COLORS[j],
        'n':           int(km_sizes[cid]),
    }
for j, cid in enumerate(km_pc2_arm):
    lbl = f'{KM_DATASET_LABEL}_{km_letters[len(km_top_chain) + j]}'
    USER_REGIONS_KM[lbl] = {
        'space':       KM_SPACE_NAME,
        'pc1_range':   (km_pc1_corner, km_pc1_max),
        'pc2_range':   (km_pc2_arm_edges[j + 1], km_pc2_arm_edges[j]),
        'description': f'PC2-arm cluster {km_letters[len(km_top_chain) + j]}',
        'color':       KM_COLORS[len(km_top_chain) + j],
        'n':           int(km_sizes[cid]),
    }

# ── Report ────────────────────────────────────────────────────────────
print('=' * 54)
print(f'K-MEANS BOUNDARY FINDER  (k={K},  space={KM_SPACE_NAME})')
print('=' * 54)
print('Cluster sizes: ' + '  '.join(
    f'{km_letters[j]}={int(km_sizes[km_ordered_ids[j]])}'
    for j in range(K)))
print(f'PC1 splits     : {km_pc1_splits}')
print(f'PC2 floors     : {km_pc2_bottoms}  (one per PC1 band)')
print(f'PC2 arm boundary: {km_bulk_pc2_boundary}')
print(f'Generated {len(USER_REGIONS_KM)} region(s).')

## Step K2 — Diagnostic plots

Two panels are produced:

- **Left:** scatter coloured by cluster label with detected boundary lines overlaid (red dashed). Verify that boundaries sit in genuine gaps between cluster groups.
- **Right:** the final rectangular region rectangles drawn over all data points.

If a boundary cuts through a dense region, increase `K` and re-run Step K1.
Save path: `{OUTPUT_DIR}/{DATASET_NAME}_{KM_SPACE_NAME}_kmeans_boundaries.png`

In [ ]:
from matplotlib.patches import Rectangle as _Rect

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: cluster assignments with boundary lines
ax = axes[0]
for j, cid in enumerate(km_ordered_ids):
    mask = km_labels == cid
    ax.scatter(km_pc1[mask], km_pc2[mask], s=10, alpha=0.4,
               color=KM_COLORS[j],
               label=f'{km_letters[j]} (n={km_sizes[cid]})')
ax.scatter(km_centroids[:, 0], km_centroids[:, 1],
           s=60, c='black', marker='x', zorder=5, label='centroids')
for b in km_pc1_splits:
    ax.axvline(b, color='red', lw=1.5, linestyle='--')
for j in range(len(km_top_chain)):
    ax.hlines(km_pc2_bottoms[j], km_pc1_edges[j], km_pc1_edges[j+1],
              colors='red', lw=1.5, linestyles='--')
for b in km_pc2_inner_splits:
    ax.hlines(b, km_pc1_corner, km_pc1_max, colors='red', lw=1.5, linestyles='--')
ax.set_xlabel('PC1', fontsize=12)
ax.set_ylabel('PC2', fontsize=12)
ax.set_title(f'{KM_DATASET_LABEL} — cluster assignments (k={K})', fontsize=12)
ax.legend(fontsize=9, markerscale=2)
ax.grid(alpha=0.3)

# Right: rectangular region overlay
ax2 = axes[1]
ax2.scatter(km_pc1, km_pc2, s=4, color='lightgrey', alpha=0.3, zorder=1)
for _lbl, _reg in USER_REGIONS_KM.items():
    x0, x1 = _reg['pc1_range']
    y0, y1 = _reg['pc2_range']
    rect = _Rect((x0, y0), x1 - x0, y1 - y0,
                 linewidth=1.5, edgecolor=_reg['color'],
                 facecolor=_reg['color'], alpha=0.15, zorder=2)
    ax2.add_patch(rect)
    short = _lbl.split('_')[-1]
    ax2.text((x0 + x1) / 2, (y0 + y1) / 2, short,
             ha='center', va='center', fontsize=12,
             color=_reg['color'], fontweight='bold')
ax2.set_xlim(km_pc1_min - 0.02, km_pc1_max + 0.02)
ax2.set_ylim(km_pc2_min - 0.02, km_pc2_max + 0.02)
ax2.set_xlabel('PC1', fontsize=12)
ax2.set_ylabel('PC2', fontsize=12)
ax2.set_title(f'{KM_DATASET_LABEL} — rectangular regions', fontsize=12)
ax2.grid(alpha=0.3)

plt.suptitle(
    f'{DATASET_NAME} — {KM_SPACE_NAME.capitalize()} space: K-means regions',
    fontsize=13)
plt.tight_layout()
km_save_path = os.path.join(OUTPUT_DIR,
    f'{DATASET_NAME}_{KM_SPACE_NAME}_kmeans_boundaries.png')
plt.savefig(km_save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {km_save_path}')

## Step K3 — Print USER_REGIONS_KM as copy-pasteable Python

Copy the output into your `behavioral_region_explorer.py` session,
renaming the region keys if desired (e.g. replace `_A`, `_B` with descriptive labels).

In [ ]:
print('# ── paste into behavioral_region_explorer.py ──────────────────')
print('USER_REGIONS = {')
for _lbl, _reg in USER_REGIONS_KM.items():
    print(f"    '{_lbl}': {{")
    print(f"        'space':       '{_reg['space']}',")
    print(f"        'pc1_range':   ({_reg['pc1_range'][0]:.6f}, {_reg['pc1_range'][1]:.6f}),")
    print(f"        'pc2_range':   ({_reg['pc2_range'][0]:.6f}, {_reg['pc2_range'][1]:.6f}),")
    print(f"        'description': '{_reg['description']}',  # n={_reg['n']}")
    print(f"        'color':       '{_reg['color']}',")
    print( "    },")
print('}')

## Step K4 — Run behavioral_region_explorer.py  *(optional)*

Fill in the dataset variables below and uncomment `%run` to run the full
analysis directly. `USER_REGIONS` is reassigned to `USER_REGIONS_KM` so
`behavioral_region_explorer.py` picks up the k-means regions automatically.

In [ ]:
# Fill in dataset variables then uncomment %run.
# USER_REGIONS is set to USER_REGIONS_KM automatically below — no duplication needed.

# DATA_FILE     = 'my_data.csv'
# ID_COLUMN     = 'ID'
# DROP_COLUMNS  = ['col1', 'col2']
# LABEL_COLUMNS = ['label1', 'label2']
# PLOT_MODE     = 'combined'
# USER_REGIONS  = USER_REGIONS_KM   # use k-means regions for the run

# %run -i behavioral_region_explorer.py